In [1]:
import pandas as pd

In [2]:
#sample dataset

data={
    'name': ['ali','sara','john',None,'Mary'],
    'age': [25,None,30,22,None],
    'city': ['kl','penang',None,'johor','kl']
}

In [4]:
df=pd.DataFrame(data)
print('original DataFrame:')
print(df)

original DataFrame:
   name   age    city
0   ali  25.0      kl
1  sara   NaN  penang
2  john  30.0    None
3  None  22.0   johor
4  Mary   NaN      kl


# check the missing value row by row

In [6]:
for index, row in df.iterrows():
    print(f'row {index}: missing={row.isnull().any()}, details={row.isnull().to_dict()}')

row 0: missing=False, details={'name': False, 'age': False, 'city': False}
row 1: missing=True, details={'name': False, 'age': True, 'city': False}
row 2: missing=True, details={'name': False, 'age': False, 'city': True}
row 3: missing=True, details={'name': True, 'age': False, 'city': False}
row 4: missing=True, details={'name': False, 'age': True, 'city': False}


In [10]:
# show only rows with missing data
missing_rows=df[df.isnull().any(axis=1)]
print('rows with missing data:')
print(missing_rows)

rows with missing data:
   name   age    city
1  sara   NaN  penang
2  john  30.0    None
3  None  22.0   johor
4  Mary   NaN      kl


In [19]:
# stimulate rows that would be dropped using dropna()
to_drop=df[df.isnull().any(axis=1)]
print('these rows would ne dropped using dropna():')
print(to_drop)

these rows would ne dropped using dropna():
   name   age    city
1  sara   NaN  penang
2  john  30.0    None
3  None  22.0   johor
4  Mary   NaN      kl


In [20]:
df_cleaned=df.dropna()
print('\nafter dropna():')
print(df_cleaned)


after dropna():
  name   age city
0  ali  25.0   kl


In [21]:
print('before fillna()')
print(df)

before fillna()
   name   age    city
0   ali  25.0      kl
1  sara   NaN  penang
2  john  30.0    None
3  None  22.0   johor
4  Mary   NaN      kl


In [27]:
df_sfill=df.fillna({
    'name':'unknown',
    'age': round(df['age'].mean(),2),
    'city': 'not available'
})

In [28]:
print('\nafter fillna(): age up to decimal digit')
print(df_sfill)


after fillna(): age up to decimal digit
      name    age           city
0      ali  25.00             kl
1     sara  25.67         penang
2     john  30.00  not available
3  unknown  22.00          johor
4     Mary  25.67             kl


In [29]:
import math

In [40]:
df_new=df.fillna({
    'name':'unknown',
    'age': math.floor(df['age'].mean().astype(int)),
    'city': 'not available'
})

In [41]:
print('\nafter fillna(): afloor without decimal')
print(df_new)


after fillna(): afloor without decimal
      name   age           city
0      ali  25.0             kl
1     sara  25.0         penang
2     john  30.0  not available
3  unknown  22.0          johor
4     Mary  25.0             kl


In [42]:
df_new=df.fillna({
    'name':'unknown',
    'age': math.ceil(df['age'].mean().astype(int)),
    'city': 'not available'
})

In [43]:
print('\nafter fillna(): afloor without decimal')
print(df_new)


after fillna(): afloor without decimal
      name   age           city
0      ali  25.0             kl
1     sara  25.0         penang
2     john  30.0  not available
3  unknown  22.0          johor
4     Mary  25.0             kl


In [44]:
#

# Data Cleaning Exercise
# Step 1: use day 15 students_performance_dirty.csv
# Step 2: Get basic information about dataset
# Step 3: print('Missing values per column:')
# Step 4: Check missing data line by line
# Step 5: Drop missing rows (if any)
# Step 6: Compare before and after
# step 7: fill missing values
# step 8: compare before and after

In [125]:
# Step 1: Load dataset
df = pd.read_csv('students_performance_dirty.csv')

In [126]:
# Show all rows
pd.set_option('display.max_rows', None)

# Step 1: Load dataset
df = pd.read_csv('students_performance_dirty.csv')

# Step 2: Clean invalid score values (<0 or >100)
for col in ['math_score', 'reading_score', 'final_score']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[(df[col] < 0) | (df[col] > 100), col] = pd.NA

# Step 3: Fill missing values safely (no warnings)
df_clean = df.copy()
for col in df_clean.columns:
    if df_clean[col].dtype == 'object':
        mode_val = df_clean[col].mode(dropna=True)
        if not mode_val.empty:
            df_clean[col] = df_clean[col].fillna(mode_val[0])
        else:
            df_clean[col] = df_clean[col].fillna("Unknown")
    else:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        mean_val = df_clean[col].mean(skipna=True)
        df_clean[col] = df_clean[col].fillna(mean_val if pd.notna(mean_val) else 0)

# Step 4: Round numeric columns
num_cols = ['study_hours', 'attendance_pct', 'math_score', 'reading_score', 'final_score']
df_clean[num_cols] = df_clean[num_cols].round(2)

# Step 5: Convert study_hours to hours + minutes
df_clean['study_time_hm'] = df_clean['study_hours'].apply(
    lambda x: f"{int(x)}h {int(round((x - int(x)) * 60))}m"
)

# Step 6: Verify cleanliness
print("✅ No NaN values left:", df_clean.isnull().sum().sum() == 0)
print("\n✅ Clean dataset preview:")
print(df_clean)

✅ No NaN values left: True

✅ Clean dataset preview:
    gender  study_hours  attendance_pct  math_score  reading_score  \
0     male         1.70            83.6       62.00          91.00   
1     male         2.60            91.2       82.00          60.00   
2   female         2.90            97.5       69.00          57.00   
3   female         4.80            85.7       78.00          62.00   
4     male         3.90           -10.0       64.00          95.00   
5     male         1.40            93.7       77.00          72.00   
6     male         3.41            87.1       95.00          78.00   
7    femle         3.41            70.6       59.00          70.00   
8   female         2.70            74.7       68.00          71.00   
9   female         4.20            92.4       63.00          66.00   
10    male         4.70            67.9       94.00          73.00   
11    male         4.60            86.2       63.00          78.00   
12   femle         3.41            95